In [1]:
import random
import time
# 1. Define the Server Model
class Server:
    def __init__(self, server_id):
        self.server_id = server_id
        self.active_connections = 0

    def add_connection(self):
        self.active_connections += 1

    def release_connection(self):
        if self.active_connections > 0:
            self.active_connections -= 1

    def __str__(self):
        return f"Server-{self.server_id} (Active: {self.active_connections})"

In [2]:

# 2. Define the Load Balancing Algorithms
class RoundRobinLB:
    def __init__(self, servers):
        self.servers = servers
        self.current_index = 0

    def get_next_server(self):
        # Pick the server at the current index
        server = self.servers[self.current_index]
        # Move to the next index, looping back to 0 if at the end
        self.current_index = (self.current_index + 1) % len(self.servers)
        return server

class LeastConnectionsLB:
    def __init__(self, servers):
        self.servers = servers

    def get_next_server(self):
        # Find the server with the absolute minimum active connections
        # If there's a tie, min() returns the first one it encounters
        least_loaded_server = min(self.servers, key=lambda s: s.active_connections)
        return least_loaded_server

In [4]:
# 3. The Simulation Function
def run_simulation(lb_strategy, servers, num_requests=10):
    print(f"\n--- Starting Simulation: {lb_strategy.__class__.__name__} ---")
    
    for i in range(1, num_requests + 1):
        print(f"\n[Request {i} Arrives]")
        
        # 1. Randomly simulate some previous connections finishing
        for server in servers:
            if server.active_connections > 0 and random.choice([True, False]):
                server.release_connection()
                print(f"  * A previous task finished on {server.server_id}")

        # 2. Use the Load Balancer to find the best server for the new request
        assigned_server = lb_strategy.get_next_server()
        assigned_server.add_connection()
        print(f"  -> Routed to: {assigned_server.server_id}")
        
        # 3. Print the current state of all servers
        state = " | ".join([str(s) for s in servers])
        print(f"  Current Load: [ {state} ]")
        
        # Pause slightly to simulate real-time processing (optional)
        time.sleep(0.5)

In [5]:
# 4. Execute the Practical
if __name__ == "__main__":
    # Initialize 3 servers for the experiment
    server_list_1 = [Server("A"), Server("B"), Server("C")]
    server_list_2 = [Server("X"), Server("Y"), Server("Z")]

    # Initialize the load balancers
    rr_balancer = RoundRobinLB(server_list_1)
    lc_balancer = LeastConnectionsLB(server_list_2)

    # Run the Round Robin Simulation
    run_simulation(rr_balancer, server_list_1, num_requests=8)

    # Run the Least Connections Simulation
    run_simulation(lc_balancer, server_list_2, num_requests=8)


--- Starting Simulation: RoundRobinLB ---

[Request 1 Arrives]
  -> Routed to: A
  Current Load: [ Server-A (Active: 1) | Server-B (Active: 0) | Server-C (Active: 0) ]

[Request 2 Arrives]
  -> Routed to: B
  Current Load: [ Server-A (Active: 1) | Server-B (Active: 1) | Server-C (Active: 0) ]

[Request 3 Arrives]
  * A previous task finished on A
  * A previous task finished on B
  -> Routed to: C
  Current Load: [ Server-A (Active: 0) | Server-B (Active: 0) | Server-C (Active: 1) ]

[Request 4 Arrives]
  * A previous task finished on C
  -> Routed to: A
  Current Load: [ Server-A (Active: 1) | Server-B (Active: 0) | Server-C (Active: 0) ]

[Request 5 Arrives]
  * A previous task finished on A
  -> Routed to: B
  Current Load: [ Server-A (Active: 0) | Server-B (Active: 1) | Server-C (Active: 0) ]

[Request 6 Arrives]
  * A previous task finished on B
  -> Routed to: C
  Current Load: [ Server-A (Active: 0) | Server-B (Active: 0) | Server-C (Active: 1) ]

[Request 7 Arrives]
  * A prev